<a href="https://colab.research.google.com/github/benjibrcz/deep-ltl-interp/blob/main/Timaeus_2026_Research_Scientist_Work_Test_(In_context_learning).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Timaeus \- Research Scientist \- Work Test 2026**

### **Background**

Large language models exhibit **in-context learning (ICL)**: they improve at prediction as they see more tokens within a single context, *without any weight updates*. There are several theoretical perspectives:

1. **ICL as in-context supervised learning**: Given question-answer pairs in the prompt, the model becomes more accurate at answering later questions.  
2. **ICL as in-context empirical risk minimization**: ICL occurs if the per-token loss decreases with context length.  
3. **ICL as in-context Bayesian inference**: ICL can be understood as approximate Bayesian inference over latent concepts or tasks.

You are **not expected** to be familiar with this literature.

---

### **Your Task**

Design and implement a prototype evaluation method to assess the in-context learning capabilities of transformers.

**Guiding questions:**

* What does it mean for a model to "learn" in-context? What does it mean for a model to *not* use in-context learning? What phenomena is ICL distinct from?   
* What tasks could reveal ICL? What properties should they have?  
* How do you measure ICL performance?  
* How do you control for confounds?

This is deliberately open-ended: you can focus on evaluating pretrained language models or on evaluating small transformers that you personally train from scratch on synthetic tasks; you can focus on evaluating a single model in detail or doing a broad comparative analysis; you can assume one of the three theoretical perspectives above and continue from there, or you can focus on comparing the three perspectives; etc. There is no "correct" way to approach this problem, and we recommend you spend some time thinking through options yourself. When you find yourself making choices about direction, please explain why you chose to focus on X over Y.

If you find yourself stuck, the references at the end of this document point to related work that may spark ideas. Please do not spend your entire work test reading papers.

---

### **What we’re looking for:**

* **Research thinking**: Clear problem formulation, awareness of confounds, thoughtful design  
* **Technical execution**: Working code, appropriate methods, correct implementation  
* **Communication**: Clear documentation, well-organized notebooks, interpretable results  
* **Depth vs. breadth**: Good decisions about what to pursue given time constraints

Feel free to decide to go deeper in some areas at the expense of others.


# 1. Design

## Overview

We evaluate ICL by presenting models with **k demonstration pairs** `(x_i, f(x_i))` followed by a
**test query** `x_test`, and measuring how well the model predicts `f(x_test)` as k increases.

## Model: Pythia family (EleutherAI)
- Available in sizes 70M → 6.9B (scaling analysis)
- Freely available, Colab-friendly
- Well-studied in ICL literature

## Tasks (synthetic, novel, ground-truth available)
1. **Arbitrary symbol mapping**: word → random integer
2. **Linear functions**: y = ax + b
3. **Modular arithmetic**: (a+b) mod p

## Confound controls (5 conditions per task)
| Condition | Description | Controls for |
|-----------|-------------|-------------|
| Standard | k correct demos | Baseline ICL |
| Irrelevant demos | k demos from different rule | "More context = better" |
| Shuffled labels | Same inputs, random outputs | Format imitation |
| Reversed order | Correct demos, reversed | Order sensitivity |
| Recency conflict | First half correct, second half wrong rule | Recency bias |

## Metrics (3 theoretical perspectives)
1. **Supervised learning**: Accuracy vs k
2. **ERM**: Per-token loss at answer position vs context length
3. **Bayesian**: Learning curve shape, permutation sensitivity

In [ ]:
# Setup and imports
!pip install -q transformers accelerate

import torch
import numpy as np
import random
from transformers import AutoModelForCausalLM, AutoTokenizer
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

In [ ]:
# Load model and tokenizer
# Start with Pythia-410M as a reasonable middle ground for testing.
# Can scale up/down later for comparative analysis.

MODEL_NAME = "EleutherAI/pythia-410m"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16 if device.type == "cuda" else torch.float32,
).to(device)
model.eval()

print(f"Loaded {MODEL_NAME} ({sum(p.numel() for p in model.parameters()) / 1e6:.0f}M params)")

# 2. Metrics

Three metrics corresponding to the three theoretical perspectives on ICL:

1. **Accuracy vs k** (supervised learning view): Does the model get the right answer more often with more demos?
2. **Answer loss vs k** (ERM view): Does the cross-entropy loss on the answer tokens decrease with more demos?
3. **Bayesian diagnostics**: Is the learning curve log-linear? Is performance invariant to demo order?

In [ ]:
@torch.no_grad()
def evaluate_prompt(model, tokenizer, prompt: str, target: str, device) -> dict:
    """Evaluate a single prompt+target pair.

    Returns:
        dict with:
            - 'accuracy': 1.0 if greedy-decoded answer matches target, else 0.0
            - 'loss': cross-entropy loss on the target tokens
            - 'generated': the model's greedy-decoded answer (for debugging)
    """
    # Tokenize prompt and target separately so we know where the answer starts
    prompt_ids = tokenizer.encode(prompt, return_tensors="pt").to(device)
    target_ids = tokenizer.encode(target, add_special_tokens=False, return_tensors="pt").to(device)
    full_ids = torch.cat([prompt_ids, target_ids], dim=1)

    # Forward pass on full sequence
    outputs = model(full_ids)
    logits = outputs.logits  # (1, seq_len, vocab)

    # Loss on target tokens only:
    # logits at position [prompt_len-1 .. prompt_len+target_len-2] predict
    # tokens at position [prompt_len .. prompt_len+target_len-1]
    prompt_len = prompt_ids.shape[1]
    target_len = target_ids.shape[1]

    target_logits = logits[0, prompt_len - 1 : prompt_len + target_len - 1, :]  # (target_len, vocab)
    target_tokens = target_ids[0]  # (target_len,)

    loss = torch.nn.functional.cross_entropy(target_logits.float(), target_tokens).item()

    # Greedy decode: generate target_len tokens from prompt
    generated_ids = model.generate(
        prompt_ids,
        max_new_tokens=target_len,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    generated_text = tokenizer.decode(generated_ids[0, prompt_len:], skip_special_tokens=True).strip()
    accuracy = 1.0 if generated_text == target.strip() else 0.0

    return {
        "accuracy": accuracy,
        "loss": loss,
        "generated": generated_text,
    }


# Quick sanity check
result = evaluate_prompt(model, tokenizer, "The capital of France is", " Paris", device)
print(f"Sanity check - generated: '{result['generated']}', loss: {result['loss']:.3f}, acc: {result['accuracy']}")

In [ ]:
# 3. Datasets

## Task 1: Arbitrary Symbol Mapping

**Setup**: Map common English words to random single-digit integers (0-9).
Each "task instance" is a fresh random mapping. The model sees k demo pairs
like `"apple -> 7\nbanana -> 3\n"` and must predict the output for a held-out word.

**Why single digits?** Keeps the target to a single token, making accuracy/loss
clean to measure. The mapping is completely arbitrary — no semantic relationship
between words and numbers — so the model can't use pretraining knowledge.

**Word pool**: We use 50 common, unambiguous English nouns. For each task instance,
we sample a subset and assign random labels.

In [ ]:
# Word pool: common, concrete, unambiguous English nouns
WORD_POOL = [
    "apple", "tiger", "river", "piano", "cloud", "bread", "chair", "flame",
    "grape", "horse", "knife", "lemon", "mouse", "ocean", "pearl", "queen",
    "robot", "snake", "torch", "whale", "arrow", "badge", "candy", "drum",
    "eagle", "fence", "globe", "heart", "ivory", "jewel", "koala", "lunar",
    "maple", "nerve", "olive", "pilot", "quilt", "reign", "storm", "tower",
    "union", "vapor", "wrist", "yacht", "zebra", "brick", "crane", "delta",
    "frost", "grain",
]

# Number of k values to test
K_VALUES = [0, 1, 2, 4, 8, 16, 32]


class SymbolMappingTask:
    """Generates task instances for arbitrary word -> digit mapping."""

    def __init__(self, word_pool=WORD_POOL, n_labels=10, seed=None):
        self.word_pool = word_pool
        self.n_labels = n_labels  # digits 0-9
        self.rng = np.random.RandomState(seed)

    def sample_task(self, n_demos: int, n_test: int = 1):
        """Sample a task instance: a random mapping and demo/test split.

        Returns:
            demos: list of (word, digit_str) pairs
            tests: list of (word, digit_str) pairs
        """
        n_total = n_demos + n_test
        words = self.rng.choice(self.word_pool, size=n_total, replace=False).tolist()
        labels = self.rng.randint(0, self.n_labels, size=n_total).tolist()

        pairs = [(w, str(l)) for w, l in zip(words, labels)]
        demos = pairs[:n_demos]
        tests = pairs[n_demos:]
        return demos, tests

    def sample_distractor_task(self, n_demos: int):
        """Sample demos from a DIFFERENT random mapping (for irrelevant-demos control)."""
        # Just sample a completely fresh set — different words, different labels
        words = self.rng.choice(self.word_pool, size=n_demos, replace=False).tolist()
        labels = self.rng.randint(0, self.n_labels, size=n_demos).tolist()
        return [(w, str(l)) for w, l in zip(words, labels)]


def format_demos(demos: list, test_word: str) -> str:
    """Format demo pairs + test query into a prompt string.

    Format:
        apple -> 7
        banana -> 3
        cherry ->
    """
    lines = [f"{word} -> {label}" for word, label in demos]
    lines.append(f"{test_word} ->")
    return "\n".join(lines)


# Quick test
task_gen = SymbolMappingTask(seed=0)
demos, tests = task_gen.sample_task(n_demos=4, n_test=1)
test_word, test_label = tests[0]
prompt = format_demos(demos, test_word)
print("Example prompt:")
print(prompt)
print(f"\nExpected answer: {test_label}")

In [ ]:
# 4. Experiments

## Task 1: Arbitrary Symbol Mapping

We run 5 experimental conditions across k = {0, 1, 2, 4, 8, 16, 32} demos:

1. **Standard**: correct demos in order → measures baseline ICL
2. **Irrelevant demos**: demos from a *different* mapping → controls for "more context helps"
3. **Shuffled labels**: same words, randomized labels → controls for format imitation
4. **Reversed order**: correct demos in reverse → tests order sensitivity
5. **Recency conflict**: first half correct, second half from wrong mapping → tests recency bias

In [ ]:
def build_conditions(task_gen: SymbolMappingTask, k: int):
    """Build all 5 experimental conditions for a given k.

    Returns a list of (condition_name, prompt, target) tuples.
    Each call samples a fresh task instance so conditions share the same
    underlying mapping and test query.
    """
    if k == 0:
        # For k=0, only standard condition makes sense (no demos to manipulate)
        _, tests = task_gen.sample_task(n_demos=0, n_test=1)
        test_word, test_label = tests[0]
        prompt = format_demos([], test_word)
        return [("standard", prompt, f" {test_label}")]

    # Sample a task instance
    demos, tests = task_gen.sample_task(n_demos=k, n_test=1)
    test_word, test_label = tests[0]
    target = f" {test_label}"

    conditions = []

    # 1. Standard: correct demos in original order
    prompt = format_demos(demos, test_word)
    conditions.append(("standard", prompt, target))

    # 2. Irrelevant demos: demos from a completely different mapping
    irrel_demos = task_gen.sample_distractor_task(k)
    prompt = format_demos(irrel_demos, test_word)
    conditions.append(("irrelevant", prompt, target))

    # 3. Shuffled labels: same words, but labels randomly reassigned
    shuffled_labels = [str(task_gen.rng.randint(0, 10)) for _ in demos]
    shuffled_demos = [(w, l) for (w, _), l in zip(demos, shuffled_labels)]
    prompt = format_demos(shuffled_demos, test_word)
    conditions.append(("shuffled_labels", prompt, target))

    # 4. Reversed order: correct demos, reversed
    reversed_demos = list(reversed(demos))
    prompt = format_demos(reversed_demos, test_word)
    conditions.append(("reversed", prompt, target))

    # 5. Recency conflict: first half correct, second half from wrong mapping
    half = k // 2
    if half > 0:
        wrong_demos = task_gen.sample_distractor_task(k - half)
        conflict_demos = demos[:half] + wrong_demos
        prompt = format_demos(conflict_demos, test_word)
        conditions.append(("recency_conflict", prompt, target))

    return conditions


def run_experiment(model, tokenizer, device, task_gen, k_values, n_trials=50):
    """Run the full experiment across all k values and conditions.

    Args:
        n_trials: number of fresh task instances per (k, condition) pair

    Returns:
        results: dict mapping condition_name -> {k -> list of result dicts}
    """
    results = defaultdict(lambda: defaultdict(list))

    for k in tqdm(k_values, desc="k values"):
        for trial in tqdm(range(n_trials), desc=f"k={k} trials", leave=False):
            conditions = build_conditions(task_gen, k)

            for cond_name, prompt, target in conditions:
                result = evaluate_prompt(model, tokenizer, prompt, target, device)
                result["k"] = k
                result["trial"] = trial
                results[cond_name][k].append(result)

    return dict(results)


# Run with a small number of trials first to verify everything works
task_gen = SymbolMappingTask(seed=42)
results = run_experiment(model, tokenizer, device, task_gen, k_values=[0, 1, 2, 4], n_trials=5)

# Print a quick summary
for cond in results:
    for k in sorted(results[cond].keys()):
        accs = [r["accuracy"] for r in results[cond][k]]
        losses = [r["loss"] for r in results[cond][k]]
        print(f"{cond:20s} k={k:2d}: acc={np.mean(accs):.2f}, loss={np.mean(losses):.3f}")

In [ ]:
# 5. Results

In [ ]:
def plot_results(results, metric="loss", title_suffix=""):
    """Plot a metric across conditions and k values.

    Args:
        results: output of run_experiment
        metric: 'accuracy' or 'loss'
        title_suffix: appended to plot title
    """
    fig, ax = plt.subplots(1, 1, figsize=(8, 5))

    colors = {
        "standard": "#2196f3",
        "irrelevant": "#f44336",
        "shuffled_labels": "#ff9800",
        "reversed": "#9c27b0",
        "recency_conflict": "#4caf50",
    }
    markers = {
        "standard": "o",
        "irrelevant": "x",
        "shuffled_labels": "s",
        "reversed": "^",
        "recency_conflict": "D",
    }

    for cond_name, k_results in results.items():
        ks = sorted(k_results.keys())
        means = [np.mean([r[metric] for r in k_results[k]]) for k in ks]
        sems = [np.std([r[metric] for r in k_results[k]]) / np.sqrt(len(k_results[k])) for k in ks]

        ax.errorbar(
            ks, means, yerr=sems,
            label=cond_name,
            color=colors.get(cond_name, "gray"),
            marker=markers.get(cond_name, "o"),
            capsize=3,
            linewidth=2,
        )

    ax.set_xlabel("Number of demonstrations (k)")
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f"Symbol Mapping: {metric.capitalize()} vs k {title_suffix}")
    ax.legend()
    ax.grid(True, alpha=0.3)

    if metric == "accuracy":
        ax.axhline(y=0.1, color="gray", linestyle="--", alpha=0.5, label="chance (1/10)")
        ax.set_ylim(-0.05, 1.05)
        ax.legend()

    plt.tight_layout()
    plt.show()


# Plot the quick test results
plot_results(results, metric="accuracy", title_suffix="(quick test)")
plot_results(results, metric="loss", title_suffix="(quick test)")

In [ ]:
# Full run with all k values and more trials
# (Uncomment and run once the quick test looks correct)

# task_gen_full = SymbolMappingTask(seed=123)
# results_full = run_experiment(
#     model, tokenizer, device, task_gen_full,
#     k_values=K_VALUES,  # [0, 1, 2, 4, 8, 16, 32]
#     n_trials=50,
# )
# plot_results(results_full, metric="accuracy", title_suffix="(n=50)")
# plot_results(results_full, metric="loss", title_suffix="(n=50)")

In [ ]:
# 6. Interpretation

# Placeholder — fill in after seeing results.
# Key questions to address:
#
# 1. Does accuracy increase / loss decrease with k for the STANDARD condition?
#    -> If yes: evidence of ICL. If no: model may be too small.
#
# 2. Does the IRRELEVANT condition also improve with k?
#    -> If yes: "more context helps" confound, not true ICL.
#    -> If no: improvement is specific to relevant demos.
#
# 3. Does SHUFFLED LABELS perform like standard or like irrelevant?
#    -> Like standard: model is imitating format, not learning the mapping.
#    -> Like irrelevant: model needs correct labels to improve.
#
# 4. Does REVERSED perform similarly to standard?
#    -> If yes: order doesn't matter much (Bayesian-like).
#    -> If worse: recency/position effects.
#
# 5. RECENCY CONFLICT: does the model follow the correct (early) or wrong (recent) demos?
#    -> Following recent: recency bias dominates.
#    -> Following early: robust evidence integration.